# 03 — Explainability (SHAP)

Loads a trained pipeline (run `02_model_training_tuning.ipynb` or `scripts/run_pipeline.py` first so `models/xgboost_tuned.joblib` exists) and generates global (bar + beeswarm) and local (per-prediction waterfall) SHAP explanations via `src/dac/explainability/shap_explain.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import joblib
from sklearn.model_selection import train_test_split

from dac.config import CONFIG
from dac.data.loader import load_uci_credit
from dac.explainability.shap_explain import explain_model
from dac.features.engineering import engineer_uci_credit_features, split_feature_columns

uci_cfg = CONFIG["data"]["uci_credit"]
df, _ = load_uci_credit()
df = engineer_uci_credit_features(df)
exclude_cols = [uci_cfg["id_col"], *uci_cfg["protected_attributes"]]
numeric_cols, categorical_cols = split_feature_columns(df, uci_cfg["target_col"], exclude_cols)
X = df[numeric_cols + categorical_cols]
y = df[uci_cfg["target_col"]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG["split"]["test_size"], random_state=CONFIG["seed"], stratify=y
)
pipeline = joblib.load(CONFIG["paths"]["models_dir"] / "xgboost_tuned.joblib")

In [ ]:
importance = explain_model(
    pipeline, X_train, X_test, "xgboost_tuned",
    CONFIG["paths"]["figures_dir"] / "uci_credit" / "shap",
)
importance.head(20)

Credit-utilization ratios and delinquency-status aggregates (`MAX_DELINQUENCY`, `MONTHS_DELINQUENT`, `BILL_LIMIT_RATIO`) are expected to dominate global importance — a useful sanity check against established credit-scoring theory, and the reference point for the SHAP-vs-EBM comparison in notebook 05.